# POE_DETAIL - Silver Layer

This notebook cleans and validates the MIMIC-IV `poe_detail` Bronze table before storing it in the Silver layer.

`POE_DETAIL` contains additional field-level information for records in the main `POE` table, using `poe_id` to link each detail back to its parent order.

The Silver layer focuses on missing values, duplicate detail records, text consistency and confirming the relationship back to `POE`.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

poe_detail = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.poe_detail_raw"
)

print(f"Rows: {poe_detail.count():,}")
print(f"Columns: {len(poe_detail.columns)}")

# Check missing values
check_columns = [
    "poe_id",
    "poe_seq",
    "subject_id",
    "field_name",
    "field_value"
]

total = poe_detail.count()

for column in check_columns:
    nulls = poe_detail.filter(F.col(column).isNull()).count()

    if nulls > 0:
        pct = (nulls / total) * 100
        print(f"{column}: {nulls:,} nulls ({pct:.2f}%)")
        

### Missing Value Findings

The main `POE_DETAIL` fields are complete, including the order ID, sequence, patient ID and field-level detail values.

The next step is to check for duplicate detail records and confirm that every `poe_id` still links back to the Silver `POE` table.

In [0]:
# Check duplicate details and POE relationship
duplicate_details = (
    poe_detail
    .groupBy(
        "poe_id",
        "poe_seq",
        "subject_id",
        "field_name",
        "field_value"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

poe = spark.table("workspace.silver.poe")

unmatched_poe = (
    poe_detail
    .select("poe_id")
    .distinct()
    .join(
        poe.select("poe_id").distinct(),
        on="poe_id",
        how="left_anti"
    )
    .count()
)

print(f"Potential duplicate detail records: {duplicate_details:,}")
print(f"Unmatched poe_id values: {unmatched_poe:,}")

### Data Quality Findings

- No duplicate detail records were found.
- All `poe_id` values successfully match the Silver `POE` table.
- The key fields are complete, so only light text standardisation is needed before writing the Silver table.

In [0]:
# Clean text fields
poe_detail_silver = (
    poe_detail
    .withColumn(
        "field_name",
        F.when(F.trim(F.col("field_name")) == "", None)
         .otherwise(F.trim(F.col("field_name")))
    )
    .withColumn(
        "field_value",
        F.when(F.trim(F.col("field_value")) == "", None)
         .otherwise(F.trim(F.col("field_value")))
    )
)

print(f"POE_DETAIL Silver: {poe_detail_silver.count():,} rows")
display(poe_detail_silver.limit(10))

### Silver Transformation Check

`POE_DETAIL` has been lightly standardised while keeping the original field-level order details intact.

The table has complete key fields, no duplicate detail records and valid links back to the Silver `POE` table.

In [0]:
# Save POE_DETAIL Silver table
(
    poe_detail_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.poe_detail")
)

bronze_count = spark.table("workspace.bronze.poe_detail_raw").count()
silver_count = spark.table("workspace.silver.poe_detail").count()

print(f"Bronze: {bronze_count:,}")
print(f"Silver: {silver_count:,}")
print(f"Match: {bronze_count == silver_count}")